In [ ]:
# Cell 1
import json
import os
from pathlib import Path
from sqlalchemy import create_engine, text

from premodel2_ver210 import GEOScorer

# 스코어러 객체 생성 (전역 모델 로딩)
scorer = GEOScorer()
print("✅ GEOScorer 모델 로드 완료!")

In [ ]:
import json
import os
from pathlib import Path
from sqlalchemy import create_engine, text

def fetch_data_from_db(page_id: int):
    # 1. 환경변수 파일(.env) 로드
    CURRENT_DIR = Path.cwd()
    ENV_PATH = CURRENT_DIR / "../project_db/geo.env"

    if ENV_PATH.exists():
        with open(ENV_PATH, "r", encoding="utf-8-sig") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if "=" in line:
                    key, val = line.split("=", 1)
                    os.environ[key.strip()] = val.strip()

    # 2. DB 연결 정보 추출
    DB_USER = os.getenv("DB_USER")
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    DB_HOST = os.getenv("DB_HOST")
    DB_PORT = os.getenv("DB_PORT")
    DB_NAME = os.getenv("DB_NAME")

    ENGINE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(ENGINE_URL)

    # 3. DB 데이터 조회
    with engine.connect() as conn:
        # (1) raw_data_table (PK: page_id)
        product_query = text("""
            SELECT 
                product_name, 
                product_cat, 
                text_contents, 
                image_text, 
                json_ld_contents 
            FROM raw_data_table 
            WHERE page_id = :page_id
        """)
        raw_product = conn.execute(product_query, {"page_id": page_id}).mappings().fetchone()

        if not raw_product:
            raise ValueError(f"page_id가 {page_id}인 상품을 raw_data_table에서 찾을 수 없습니다.")

        raw_product = dict(raw_product)

        # json_ld_contents 파싱
        json_ld_data = raw_product['json_ld_contents']
        if isinstance(json_ld_data, str) and json_ld_data.strip():
            try:
                json_ld_data = json.loads(json_ld_data)
            except Exception:
                pass

        # (2) question_table (PK: query_id)
        questions_query = text("""
            SELECT 
                query_id,
                query_text, 
                query_cat, 
                query_keyword 
            FROM question_table
        """)
        raw_questions = conn.execute(questions_query).mappings().fetchall()

        user_queries = []
        for q in raw_questions:
            keywords = q['query_keyword']
            if isinstance(keywords, str):
                try:
                    keywords = json.loads(keywords)
                except Exception:
                    keywords = [x.strip() for x in keywords.split(',') if x.strip()]
            elif keywords is None:
                keywords = []

            user_queries.append({
                "query_text": q['query_text'],
                "category": q['query_cat'],
                "must_have": keywords
            })

        # (3) image_data_table (FK: page_id / PK: page_id, image_sequence)
        image_query = text("""
            SELECT 
                alt_contents 
            FROM image_data_table 
            WHERE page_id = :page_id
            ORDER BY image_sequence ASC
        """)
        raw_images = conn.execute(image_query, {"page_id": page_id}).mappings().fetchall()

        image_list = [
            {
                "alt": img['alt_contents'] or "",
                "is_text_image": False
            }
            for img in raw_images
        ]

        return raw_product, user_queries, json_ld_data, image_list

In [ ]:
# 진단할 대상의 page_id 입력
TARGET_PAGE_ID = 101

# DB 추출
raw_product, user_queries, json_ld, image_list = fetch_data_from_db(TARGET_PAGE_ID)

# 스코어러 실행
result = scorer.evaluate_page(
    body_text=raw_product['text_contents'] or "",
    image_text=raw_product['image_text'] or "",
    user_queries=user_queries,
    product_cat=raw_product['product_cat'],
    product_name=raw_product['product_name'],
    json_ld_str=json_ld,
    image_list=image_list
)